In [3]:
import pandas as pd
from ydata_profiling import ProfileReport
import datamart_profiler
import openai
from dotenv import load_dotenv
from openai import OpenAI
import os
import glob
import json
import sys
sys.path.append(r'c:\Users\ricca\Documents\Polimi\tesi\code')
import utils
import warnings
warnings.filterwarnings('ignore')

from tqdm import tqdm  # For progress bar
from multiprocessing import Pool

import importlib
importlib.reload(utils)


<module 'utils' from 'c:\\Users\\ricca\\Documents\\Polimi\\tesi\\code\\utils.py'>

In [4]:
csv_files = glob.glob('csv_files/*.csv')
print(f"Found {len(csv_files)} csv files")

Found 7 csv files


We generate the data profiles for the diabetes datasets

In [5]:
data_profiles = []
for file_path in tqdm(csv_files, desc="Generating Data Profiles"):
    try:
        profile = utils.generate_data_profile(file_path)
        data_profiles.append(profile)
    except Exception as e:
        print(f"Error processing {file_path}: {str(e)}")

print(f"\nProcessed {len(data_profiles)} files successfully")

Generating Data Profiles:  43%|████▎     | 3/7 [08:32<11:23, 170.98s/it]


KeyboardInterrupt: 

We generate the semantic profiles for the diabetes datasets

In [6]:
semantic_profiles = []

for data_profile in data_profiles:
    prompt = utils.generate_prompt(data_profile, utils.TEMPLATE_SEMANTIC, utils.RESPONSE_EXAMPLE_SEMANTIC)
    response = utils.call_openai_api(prompt, "o3-mini")
    semantic_profiles.append(response.choices[0].message.content)

In [7]:
dataset_info = [
    {
        "dataset_name": title,
        "data_profile": data_profiles[i],
        "semantic_profile": semantic_profiles[i],
        "instructions": None
    }
    for i, title in enumerate(csv_files)
]

In [17]:
for dataset in dataset_info:
    prompt = utils.generate_prompt_instructions(dataset.get("dataset_name"), dataset.get("data_profile"), dataset.get("semantic_profile"))
    dataset["instructions"] = utils.json_to_dict(utils.call_openai_api(prompt, "o1-mini").choices[0].message.content)


In [18]:
dataset_info[6].get("instructions")

{'queries': [{'query': 'Find a healthcare dataset related to gestational diabetes.'},
  {'query': 'Show me datasets that include patient attributes like age, weight, and BMI for diabetes prediction.'},
  {'query': 'Locate a dataset used for predictive modeling of gestational diabetes.'},
  {'query': 'Search for a dataset with patient information including number of pregnancies and heredity factors related to gestational diabetes.'},
  {'query': 'Find datasets in the healthcare domain that focus on feature analysis for gestational diabetes.'},
  {'query': 'Show me a dataset containing numeric data on age, weight, height, and BMI for predicting gestational diabetes.'},
  {'query': 'Locate a patient attribute dataset used for machine learning models predicting gestational diabetes.'},
  {'query': 'Search for gestational diabetes datasets that include a prediction column indicating diabetes outcomes.'},
  {'query': 'Find a dataset without temporal or spatial data that focuses on gestationa

A questo punto ci serve definire un vector db da riempire con le varie instructions

In [5]:
%pip install -q onnxruntime

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: C:\Users\ricca\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import chromadb

In [6]:
try:
    chroma_client = chromadb.PersistentClient()
    chroma_client.get_settings().allow_reset = True
    #chroma_client.reset()
except ValueError as e:
    print(f"Resetting existing Chroma instance: {e}")
    #chroma_client.reset()

embedding_function = utils.OpenAIEmbeddingFunction()

collection = chroma_client.get_or_create_collection(
    name="instructions",
    embedding_function=embedding_function
)

Resetting existing Chroma instance: An instance of Chroma already exists for ./chroma with different settings


In [20]:
for dataset in dataset_info:
    instructions = dataset.get("instructions", {}).get("queries", [])
    for i, instruction in enumerate(instructions):
        collection.add(
            documents=[instruction["query"]],
            ids=[f"instruction_{i}_{dataset.get('dataset_name')}"],
            metadatas={
                "dataset": dataset.get("dataset_name"),
                "semantic_profile": dataset.get("semantic_profile"),
                "data_profile": dataset.get("data_profile")
            }
        )

In [21]:
def query_to_vector_db(query_text, n=3):
    results = collection.query(
        query_texts=[query_text],
        n_results=n
    )

    print(f"\nQuery: {query}")
    print(f"Top {n} results:")
    for i, doc in enumerate(results['documents'][0]):
        print(f"{i+1}. {doc} (ID: {results['ids'][0][i]}, Distance: {results['distances'][0][i]:.4f})")

In [22]:
query = "I need data which contains information about the relations about pregnancy and diabetes"

query_to_vector_db(query, 10)


Query: I need data which contains information about the relations about pregnancy and diabetes
Top 10 results:
1. Locate a dataset that includes the number of pregnancies and diabetes outcomes. (ID: instruction_3_csv_files\diabetes.csv, Distance: 0.5857)
2. Search for a dataset with patient information including number of pregnancies and heredity factors related to gestational diabetes. (ID: instruction_3_csv_files\Gestational Diabetes.csv, Distance: 0.5992)
3. Find a healthcare dataset related to gestational diabetes. (ID: instruction_0_csv_files\Gestational Diabetes.csv, Distance: 0.6840)
4. Find a dataset that combines patient attributes such as Pregnancies, Glucose, and Age to predict diabetes. (ID: instruction_12_csv_files\diabetes.csv, Distance: 0.6880)
5. Show me datasets that offer numeric patient attributes for studying gestational diabetes outcomes. (ID: instruction_13_csv_files\Gestational Diabetes.csv, Distance: 0.7156)
6. Show me a dataset containing numeric data on age, 